In [1]:
#Install the relevent packages into the environment
!python -m pip install pennylane --upgrade
!python -m pip install numpy
!python -m pip install matplotlib
!python -m pip install pandas
!python -m pip install "jax<0.11"
!python -m pip install optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 61.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 55.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 97.9 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 54.9 MB/s  0:00:01
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.11.1
    Uninstalling jaxlib-0.11.1:
      Successfully uninstalled jaxlib-0.11.1
  Attempting uninstall: jax
    Found existing installation: jax 0.11.1
    Uninstalling jax-0.11.1

In [2]:
# Import required Packages partially from Reference [2]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qp
from pennylane import numpy as pnp
import jax
import io
import time
from jax import numpy as jnp
import optax
from google.colab import files
uploaded = files.upload() # Request a file to be uploaded "Surface_features.csv"

Dataset = files.upload() # Request a file to determine circuits to test "QNN_Data"
# Set jax to use 8 byte precision
jax.config.update("jax_enable_x64", True)
### After running this section. press choose file and select Surface_features.csv ###

Saving Surface_features.csv to Surface_features.csv


Saving QNN_Data.xlsx to QNN_Data.xlsx


In [3]:
Inputs = pd.read_excel(io.BytesIO(Dataset['QNN_Data.xlsx']),header=0).values
for iii in range(len(Inputs)):
  # Set number of qubits/wires
  N = int(Inputs[iii,1])
  wires = int(Inputs[iii,1]) # Set number of wires
  layers = int(Inputs[iii,3]) # Set number of layers
  Perc_Train = Inputs[iii,2]
  dev = qp.device('default.qubit', wires=N)

  print("Qubits:",N , "Test Samples:",Perc_Train , "Parameters:",(layers+1)*3*N,"Layers",layers, "Test:", iii)
  # from Reference [4]
  # Define a function for data embedding layers
  def SZ(x):
    qp.AngleEmbedding(x, wires=range(N),rotation='Z')
  def SX(x):
    qp.AngleEmbedding(x, wires=range(N),rotation='X')
  def SY(x):
    qp.AngleEmbedding(x, wires=range(N),rotation='Y')
  # Define a function for trainable layers
  def W(params):
    qp.StronglyEntanglingLayers(params, wires=range(N))
  # Import the primary input features from the csv file from Ref [1]
  Geometry = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,6,7,8,9,10,11,12)).iloc[0:45].values

  # Import the output features and its normailzation factor
  Ra_ch = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(5,)).iloc[0:45].values
  Ra_ch = Ra_ch[:,0]
  ks = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(13,)).iloc[0:45].values
  ks = ks[:,0]

  # Data reconfiguration, to be used in ML from Reference [1]

  X = Geometry
  y = np.divide(ks,Ra_ch)


  X[:,0] = np.divide(X[:,0],Ra_ch)
  X[:,2] = np.abs(X[:,2])


  # Generate secondary features and append them to the original dataset

  n,m = X.shape
  X0 = np.ones((n,1))
  X1 = np.ones((n,1))
  X2 = np.ones((n,1))
  X3 = np.ones((n,1))
  X4 = np.ones((n,1))
  X5 = np.ones((n,1))
  X6 = np.ones((n,1))
  X7 = np.ones((n,1))
  X8 = np.ones((n,1))
  X9 = np.ones((n,1))
  X10 = np.zeros((n,1))



  X1[:,0] = np.transpose(X[:,4]*X[:,5])
  X2[:,0] = np.transpose(X[:,4]*X[:,6])
  X3[:,0] = np.transpose(X[:,4]*X[:,7])
  X4[:,0] = np.transpose(X[:,5]*X[:,6])
  X5[:,0] = np.transpose(X[:,5]*X[:,7])
  X6[:,0] = np.transpose(X[:,6]*X[:,7])
  X7[:,0] = np.transpose(X[:,4]*X[:,4])
  X8[:,0] = np.transpose(X[:,5]*X[:,5])
  X9[:,0] = np.transpose(X[:,6]*X[:,6])




  X = np.hstack((X,X1))
  X = np.hstack((X,X2))
  X = np.hstack((X,X3))
  X = np.hstack((X,X4))
  X = np.hstack((X,X5))
  X = np.hstack((X,X6))
  X = np.hstack((X,X7))
  X = np.hstack((X,X8))
  X = np.hstack((X,X9))
  X = np.hstack((X,X10)) # Padding array of zeros
  X = np.hstack((X,X10)) # Padding array of zeros
  X = np.hstack((X,X10)) # Padding array of zeros

  Z2 = X

  # Normalize input features to be between 0 and pi
  for features in range(17):
    Z2[:,features] = 2*np.pi*(X[:,features] - np.min(X[:,features]))/(np.max(X[:,features])-np.min(X[:,features]))

  # Normalize output features to be between 0 and 1 with some room on either side
  Y_max = 15
  Y_min = 1
  Y = 2*(y - Y_min)/(Y_max-Y_min)-1

  # Save the sets of features used in training
  x_input = Z2
  y_input = Y
  # from Reference [2]
  @qp.qnode(dev,interface="jax") # Initialize the qubit to interface with jax
  def quantum_neural_network(params, x):
      layers=len(params[:,0,0])-1 # Input number of layers
      n_wires=len(params[0,:,0])  # number of wires
      n_params_rot=len(params[0,0,:]) # number of rotation parameters

      for j in range(N):
        qp.Hadamard(j)

      W(params[0,:,:].reshape(1,n_wires,n_params_rot))
      N_loop = int(np.ceil(17/N))
      N_set = N
      for i in range(layers):
        Set = i % N_loop

        start = N_set * Set
        end = start + N_set
        if N == 1:
          qp.RZ(x[start],0)
        else:
          SZ(x[start:end])
        W(params[i+1,:,:].reshape(1, n_wires, n_params_rot))

      observable = qp.PauliZ(0)

      for j in range(1, N):
        observable = observable @ qp.PauliZ(j)

      return qp.expval(observable)

      # from Reference [2]

  @jax.jit # uses jax's just in time (JIT) compiler
  def mse(params,x,targets): # Define a mean square error function
      # We compute the mean square error between the target function and the quantum circuit to quantify the quality of our estimator
      return jnp.abs(quantum_neural_network(params,x)-jnp.array(targets))**2 #compare the value found using the QNN to the output feature (target)
  @jax.jit # uses jax's just in time (JIT) compiler
  def loss_fn(params, x,targets): # define a loss function
      # We define the loss function to feed our optimizer
      mse_pred = jax.vmap(mse,in_axes=(None, 0,0))(params,x,targets) # Calculate mse, in_axes specifies how multiplication is performed
      loss = jnp.mean(mse_pred) # define loss as the mean mse
      return loss



  def Jax_Test_Train_Split(X, y, test_size):

      samples = X.shape[0]
      test_samples = int(round(samples * test_size))
      indices = np.random.permutation(samples)

      test_indices = indices[:test_samples]
      train_indices = indices[test_samples:]

      X_train = X[train_indices]
      X_test = X[test_indices]
      y_train = y[train_indices]
      y_test = y[test_indices]

      return X_train, X_test, y_train, y_test, test_indices



  Schedule = optax.schedules.exponential_decay(0.25, 500, 0.9, staircase=False) # Set an optimizing schedule
  opt = optax.adamw(learning_rate=Schedule) # Set the optimizing routine and its learning rate
  max_steps = 5000 # Set the number of max steps


  @jax.jit # uses jax's just in time (JIT) compiler
  def update_step_jit(i, args):

      params, opt_state, data, targets, print_training, best_params, best_loss, loss_array = args

      # Calculate loss and gradients for the adam optimization
      loss_val, grads = jax.value_and_grad(loss_fn)(params, data, targets)

      # Adam update using update and apply_updates
      updates, opt_state = opt.update(grads, opt_state, params)
      params = optax.apply_updates(params, updates)

      # Calculate loss of the updated parameters
      new_loss = loss_fn(params, data, targets)
      loss_array = loss_array.at[0,i].set(new_loss)
      # Update best parameters if new loss is lower
      def update_best():
          return params, new_loss

      def keep_best():
          return best_params, best_loss
      # Determine if the new parameters need to be saved using a coparison between the new and old loss
      best_params, best_loss = jax.lax.cond(new_loss < best_loss,update_best,keep_best)

      # Print every 10 steps if requested
      def print_fn():
          jax.debug.print("Step: {i}  Loss: {loss}", i=i,loss=new_loss)

      # Determine if the output should be printed
      jax.lax.cond((jnp.mod(i, 5000) == 0) & print_training,print_fn,lambda: None)

      # return the original arguments
      return (params,opt_state,data,targets,print_training,best_params,best_loss,loss_array)


  @jax.jit # uses jax's just in time (JIT) compiler
  def optimization_jit(params, data, targets, print_training=False):

    # Initial best values
      best_loss = jnp.inf # infinity
      best_params = params # random values
      loss_array = np.zeros([1,max_steps])
      # Initialize the optimizer
      opt_state = opt.init(params)

      # Bundle a list of variables as args
      args = (params, opt_state, jnp.asarray(data), targets, print_training, best_params, best_loss, loss_array)

      # Optimization loop
      (params, opt_state, _, _, _, best_params, best_loss, loss_array) = jax.lax.fori_loop(0,max_steps + 1,update_step_jit,args)

      return best_params, best_loss, loss_array
  # Evalutate the best set of parameters from training
  def evaluate(params, data):
      return jax.vmap(quantum_neural_network,in_axes=(None, 0))(params, jnp.asarray(data))
  # from Reference [1] and [12]
  Iterations = 100 # Number of outer loops
  miny1=1000
  miny2=1000
  miny3=1000
  params_shape = qp.StronglyEntanglingLayers.shape(n_layers=layers+1,n_wires=wires) # Determine size of parameter array
  miny = np.zeros([Iterations,3])
  # Run training
  start_time = time.perf_counter()
  for i in range(Iterations):
    print("Iteration: %f" % i)
    params = pnp.random.default_rng().random(size=params_shape) # Set random parameters

    X_train, X_test, y_train, y_test, indices = Jax_Test_Train_Split(x_input, y_input, Perc_Train)
    best_params, best_loss, loss_array = optimization_jit(params,X_train,jnp.array(y_train),print_training=True)


    y_predictions=evaluate(best_params,x_input)
    yn = (y_predictions+1)/2*(Y_max-Y_min)+Y_min
    miny[i,0] = max(100.*abs(yn-y)/y)
    miny[i,1] = np.mean(100.*abs(yn-y)/y)
    miny[i,2] = max(abs(np.log(yn/y)))
    if miny1>miny[i,0]:
      best_params_prime1 = best_params
      indices_prime1 = indices
      miny1=miny[i,0]
      print("Miny1: %f" % miny1)

    if miny2>miny[i,1]:
      best_params_prime2 = best_params
      indices_prime2 = indices
      miny2=miny[i,1]
      print("Miny2: %f" % miny2)

    if miny3>miny[i,2]:
      best_params_prime3 = best_params
      indices_prime3 = indices
      miny3=miny[i,2]
      print("Miny3: %f" % miny3)
  end_time = time.perf_counter()
  execution_time = end_time - start_time
  # From Reference [2]
  print(miny1) # Print the best loss from training
  print(miny2) # Print the best loss from training
  print(miny3) # Print the best loss from training
  y_predictions=evaluate(best_params_prime2,x_input) # Evalutate the results of training
  print("Max predicted value:", max(y_predictions))
  print("Min predicted value:", min(y_predictions))
  yn = (y_predictions+1)/2*(Y_max-Y_min)+Y_min # reverse the normalization on the output parameters

  # Write data to QNN_Data
  df_to_insert = pd.DataFrame({"Value": [np.mean(100.*abs(yn-y)/(y))]})
  with pd.ExcelWriter("QNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=5, header=False, index=False)
  df_to_insert = pd.DataFrame({"Value": [max(100.*abs(yn-y)/(y))]})
  with pd.ExcelWriter("QNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=6, header=False, index=False)
  df_to_insert = pd.DataFrame({"Value": [execution_time]})
  with pd.ExcelWriter("QNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=7, header=False, index=False)
  print("Qubits:",N , "Test Samples:",Perc_Train , "Parameters:",(layers+1)*3*N,"Layers",layers, "Mean Error:", np.mean(100.*abs(yn-y)/(y)), "Max Error:", max(100.*abs(yn-y)/(y)))
files.download('QNN_Data.xlsx')

Qubits: 2 Test Samples: 0.3 Parameters: 60 Layers 9 Test: 0
Iteration: 0.000000
Step: 0  Loss: 0.524113994973057
Step: 5000  Loss: 0.00016061505649607395
Miny1: 127.158520
Miny2: 15.812802
Miny3: 0.820478
Iteration: 1.000000
Step: 0  Loss: 0.2186367918870798
Step: 5000  Loss: 1.5850520829185338e-05
Miny2: 11.210473
Iteration: 2.000000
Step: 0  Loss: 0.4878451588195735
Step: 5000  Loss: 6.721333059399713e-07
Iteration: 3.000000
Step: 0  Loss: 0.3146846075625209
Step: 5000  Loss: 6.764055853075968e-06
Iteration: 4.000000
Step: 0  Loss: 0.24792285290849306
Step: 5000  Loss: 2.7837692318552436e-05
Iteration: 5.000000
Step: 0  Loss: 0.42081554961351364
Step: 5000  Loss: 2.0935366447301787e-06
Iteration: 6.000000
Step: 0  Loss: 0.5433112877137368
Step: 5000  Loss: 1.215110629387809e-05
Miny2: 11.140107
Iteration: 7.000000
Step: 0  Loss: 0.528331694843947
Step: 5000  Loss: 4.179526635152519e-07
Iteration: 8.000000
Step: 0  Loss: 0.42727968221901885
Step: 5000  Loss: 1.6227217565883765e-06
Ite

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>